# MatPES Workflow Comparison and Validation

The notebook is used to check that the results from the static workflows defined in run_calculation.py agree with the MatPes results.
- We load the data, define some metadata and run the two different workflows. We might want to just run both for a few thousand calculations and see which ones is faster/ more reliable
- Then we query for the data and compare to the MatPes results
- Compare the estimated relative cost to the actual relative time

In [ ]:
from run_calculation import static_calculation, static_calculation_off_equilibrium
from run_utilities import run_batch
from pymatgen.core import Structure
import json
from jobflow import SETTINGS
import numpy as np
from ase.stress import full_3x3_to_voigt_6_stress, voigt_6_to_full_3x3_stress
store = SETTINGS.JOB_STORE
store.connect()


In [ ]:
#Load example MatPES data
matpes_0 = json.load(open("../data/matpes-0.json"))
metadata_0 = {"mat_id": matpes_0.get("matpes_id"),"compare with": "matpes-0", "flow": "off_equilibrium"} 
matpes_1 = json.load(open("../data/matpes-1.json"))
metadata_1 = {"mat_id": matpes_1.get("matpes_id"),"compare with": "matpes-1", "flow": "off_equilibrium" }
batch_metdata = {"batch_name": "compare_matpes", "description": "Compare MatPES data", "flow": "off_equilibrium"}
structure_list = [Structure.from_dict(matpes_0["structure"]), Structure.from_dict(matpes_1["structure"]),]
run_batch(
    metadata_list=[metadata_0, metadata_1],
    structure_list=structure_list,
    batch_metadata=batch_metdata,
    calc_func=static_calculation_off_equilibrium,
    ncore=1,
    kpar=1,
    ignore_memory_check=True,
    worker="test_function_worker",
    nbands=[1,1],
    nkpts=[1,1],
    )

In [ ]:
# submit calculations to launchpad, possibly adjust KPAR and NCORE

static_calculation(Structure.from_dict(
    matpes_0.get("structure")),
    metadata=metadata_0,
    KPAR=4,
    NCORE=1,
    worker="euler",
)
static_calculation(Structure.from_dict(
    matpes_1.get("structure")),
    metadata=metadata_1,
    KPAR=4,
    NCORE=1,
    worker="euler",
)

static_calculation_off_equilibrium(Structure.from_dict(
    matpes_0.get("structure")),
    metadata=metadata_0,
    KPAR=4,
    NCORE=1,
    worker="euler",
)
static_calculation_off_equilibrium(Structure.from_dict(
    matpes_1.get("structure")),
    metadata=metadata_1,
    KPAR=4,
    NCORE=1,
    worker="euler",
)

In [ ]:
# Query results for the given material ID
mat_id = matpes_0.get("matpes_id")
results = list(store.query({"metadata.mat_id": mat_id}))

if not results:
    raise ValueError(f"No results found for mat_id: {mat_id}")

# Get the first result's output data
result_output = results[0]["output"]["output"]

# Compute differences
energy_diff = result_output["energy"] - matpes_0["energy"]
force_diff = np.linalg.norm(np.array(result_output["forces"]) - np.array(matpes_0["forces"]))
stress_diff = np.linalg.norm(np.array(result_output["stress"]) - voigt_6_to_full_3x3_stress(matpes_0["stress"]))
bandgap_diff = result_output["bandgap"] - matpes_0["bandgap"]

# Extract timing and metadata
total_times_0 = [res["output"]["calcs_reversed"][0]["output"]["run_stats"]["total_time"] for res in results]
metadata_list = [res["metadata"] for res in results]

# Print results
print(f"Energy difference: {energy_diff}")
print(f"Force difference (norm): {force_diff}")
print(f"Stress difference (norm): {stress_diff}")
print(f"Bandgap difference: {bandgap_diff}")
print("Total times:", total_times_0)

In [ ]:
# Query results for the given material ID
mat_id = matpes_1.get("matpes_id")
results = list(store.query({"metadata.mat_id": mat_id}))

if not results:
    raise ValueError(f"No results found for mat_id: {mat_id}")

# Get the first result's output data
result_output = results[0]["output"]["output"]

# Compute differences
energy_diff = result_output["energy"] - matpes_1["energy"]
force_diff = np.linalg.norm(np.array(result_output["forces"]) - np.array(matpes_1["forces"]))
stress_diff = np.linalg.norm(np.array(result_output["stress"]) - voigt_6_to_full_3x3_stress(matpes_1["stress"]))
bandgap_diff = result_output["bandgap"] - matpes_1["bandgap"]

# Extract timing and metadata
total_times_1 = [res["output"]["calcs_reversed"][0]["output"]["run_stats"]["total_time"] for res in results]
metadata_list = [res["metadata"] for res in results]

# Print results
print(f"Energy difference: {energy_diff}")
print(f"Force difference (norm): {force_diff}")
print(f"Stress difference (norm): {stress_diff}")
print(f"Bandgap difference: {bandgap_diff}")
print("Total times:", total_times_1)

In [ ]:
from estimate_cost import get_cost_info_from_structure
#compare total times with cost estimates
composition_0 = matpes_0["composition"]
lattice_0 = np.array(matpes_0["structure"]["lattice"]["matrix"])
spg = matpes_0["symmetry"]["number"]
Nbands_0, Kpoints_0, spg_0, N_irreducible_kpts_0, cost_estimate_0 = get_cost_info_from_structure(composition_0, lattice_0, spg)
composition_1 = matpes_1["composition"]
lattice_1 = np.array(matpes_1["structure"]["lattice"]["matrix"])
spg_1 = matpes_1["symmetry"]["number"]
Nbands_1, Kpoints_1, spg_1, N_irreducible_kpts_1, cost_estimate_1 = get_cost_info_from_structure(composition_1, lattice_1, spg_1)
relative_cost = cost_estimate_0 / cost_estimate_1
print(f"Relative cost: {relative_cost:.4f}")


relative_calc_time = total_times_0[0] / total_times_1[0]

print(f"Relative calculation time: {relative_calc_time:.4f}")
